In [ ]:
from pathlib import Path
from io import BytesIO
import json, csv, hashlib, math

PACKAGE_ROOT = next((p for p in (Path.cwd(), Path.cwd().parent)
                     if (p / "data").is_dir() and (p / "notebooks").is_dir()), None)
if PACKAGE_ROOT is None:
    raise FileNotFoundError("Open this notebook from the package root or notebooks directory.")

SAVE_PDF = False

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import t
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from IPython.display import display, Image
plt.rcdefaults()

def load_npz(relative_path):
    with np.load(PACKAGE_ROOT / relative_path, allow_pickle=False) as archive:
        return {key: archive[key].copy() for key in archive.files}



def export_figure(figure, figure_id, tight=False):
    options = {"bbox_inches": "tight"} if tight else {}
    preview = BytesIO()
    figure.savefig(preview, format="png", dpi=180, **options)
    display(Image(data=preview.getvalue()))
    if SAVE_PDF:
        output = PACKAGE_ROOT / "outputs"
        output.mkdir(exist_ok=True)
        figure.savefig(output / f"{figure_id}.pdf",
                       metadata={"CreationDate": None, "ModDate": None}, **options)
        print(f"Saved outputs/{figure_id}.pdf")


In [ ]:
ORDER_COLORS = {1: "#009E73", 2: "#0072B2", 3: "#E69F00", 4: "#D55E00", 5: "#CC79A7", 6: "#777777", 7: "#8A6A00"}
ORDER_STYLES = {1: (0, (6, 1)), 2: "-", 3: "--", 4: "-.", 5: (0, (5, 1, 1, 1)), 6: ":", 7: (0, (2, 1, 2, 1, 6, 1))}
plt.rcParams.update({
    "font.family": "serif", "font.serif": ["DejaVu Serif"],
    "font.size": 10, "axes.titlesize": 11.5, "axes.labelsize": 11,
    "xtick.labelsize": 9.5, "ytick.labelsize": 9.5, "legend.fontsize": 8.5,
    "pdf.fonttype": 42, "ps.fonttype": 42, "axes.linewidth": 0.8,
    "axes.spines.top": True, "axes.spines.right": True,
    "figure.facecolor": "white", "axes.facecolor": "white",
})
def style_axis(axis, letter, title):
    axis.set_title(title, fontweight="normal", pad=10)
    axis.text(-0.16, 1.055, f"({letter})", transform=axis.transAxes,
              fontsize=11, fontweight="bold", ha="left", va="bottom")
    axis.tick_params(direction="out", length=3.5, width=0.8)
    for spine in axis.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.8)
    axis.grid(alpha=0.14, which="major", linewidth=0.5)
    axis.set_xlim(0, 200)
    axis.set_xticks((0, 50, 100, 150, 200))

def circuit_panel(axis, task):
    axis.set_axis_off();axis.set_xlim(-.8,11);axis.set_ylim(0,4.65)
    axis.set_title("4-qubit circuit and target", fontweight="normal", pad=10)
    axis.text(-.095,1.055,"(a)",transform=axis.transAxes,fontsize=11,fontweight="bold",ha="left",va="bottom")
    rows=(3.55,2.65,1.75,.85)
    for qubit,y in enumerate(rows):
        axis.plot((.65,10.8),(y,y),color="black",lw=.8)
        axis.text(.45,y,rf"$q_{qubit}:|0\rangle$",ha="right",va="center",fontsize=8)
        for x,k in ((2.1,qubit),(6.05,qubit+4)):
            letter="Y" if k<4 else "Z"
            axis.add_patch(Rectangle((x-.91,y-.30),1.82,.60,facecolor="white",edgecolor="#0072B2",lw=.8,zorder=3))
            axis.text(x,y+.11,rf"$R_{letter}(x_{k})$",ha="center",va="center",fontsize=8,zorder=4)
            axis.text(x,y-.15,rf"$\mathcal{{N}}_{{{letter},\gamma}}$",ha="center",va="center",fontsize=7.7,zorder=4)
    for j,x in enumerate((3.4,4.2,5.0)):
        upper,lower=rows[j:j+2]
        axis.plot((x,x),(lower,upper),color="black",lw=.8)
        axis.plot((x,x),(lower,upper),"o",color="black",ms=2.6)
        axis.text(x,(upper+lower)/2,"CZ",fontsize=7,ha="center",va="center",bbox={"facecolor":"white","edgecolor":"none","pad":.4})
    axis.add_patch(Rectangle((8,.48),2.25,3.5,facecolor="white",edgecolor="#56B4E9",lw=.9,zorder=3))
    axis.text(9.125,2.2,r"$V(\boldsymbol{\theta})$",ha="center",va="center",fontsize=10,zorder=4)
    axis.text(5.2,4.36,r"$O_4=\sum_{\alpha=1}^{9}w_\alpha P_\alpha,\quad\|O_4\|_\infty\leq1$",ha="center",va="center",fontsize=8.3)
    target=(r"$y_\gamma(\mathbf{x})=F_\gamma(\mathbf{x};\mathbf{0})$" if task=="matched" else r"$y_0(\mathbf{x})=F_0(\mathbf{x};\mathbf{0})$")
    axis.text(5.2,.10,target,ha="center",va="center",fontsize=8.6)

In [ ]:
data=load_npz("data/FigS6/additional_noise.npz")
meta=json.loads((PACKAGE_ROOT/"data/FigS6/additional_noise.json").read_text())
losses=data["sector_loss"];ci=data["condition_index"];iterations=data["iteration"]
fig,axes=plt.subplots(2,3,figsize=(11.5,7.4))
fig.subplots_adjust(left=.085,right=.985,bottom=.10,top=.82,wspace=.28,hspace=.67)
handles=[]
for panel,cond in enumerate(meta["panels"]):
    row,col=divmod(panel,3);ax=axes[row,col];r=cond['r'];model=cond['model']
    if model=='fixed_z': title='Encoding fixed-axis Pauli-Z\n'+rf'$p_z={1.5*r:g}$'
    elif model=='symmetric_pauli': title='Encoding symmetric Pauli\n'+rf'$p_x=p_y=p_z={r/2:g}$'
    else: title='Encoding thermal relaxation\n'+rf'$t_g/T_1={-np.log1p(-2*r):.5f},\ T_2=T_1$'
    style_axis(ax,cond['panel'],title);ax.set_xlim(0,2000);ax.set_xticks([0,500,1000,1500,2000])
    ax.set_yscale('log');ax.set_ylim(meta['shared_y_limits'])
    take=ci==cond['source_condition'];assert take.sum()==20
    for k in range(1,8):
        q=np.quantile(losses[:,take,k]/losses[0,take,k],[.25,.5,.75],axis=1)
        ax.fill_between(iterations,q[0],q[2],color=ORDER_COLORS[k],alpha=.08,linewidth=0)
        line,=ax.plot(iterations,q[1],color=ORDER_COLORS[k],linestyle=ORDER_STYLES[k],linewidth=1.1)
        if panel==0:handles.append(line)
    if col==0:ax.set_ylabel(r'Relative sector loss $R_k(t)$')
    ax.axhline(1,color='.4',ls='--',lw=.65,zorder=0);ax.axhline(.5,color='.55',ls=':',lw=.65,zorder=0)
fig.legend(handles,[rf'$k={k}$' for k in range(1,8)],loc='upper center',bbox_to_anchor=(.535,.995),ncol=7,frameon=False,handlelength=2.5,columnspacing=1.)
fig.text(.535,.935,r'$r=0.01$',ha='center',va='center',fontsize=11.5)
fig.text(.535,.49,r'$r=0.05$',ha='center',va='center',fontsize=11.5)
fig.supxlabel('Training iteration',fontsize=11,y=.02)
export_figure(fig,'FigS6',tight=True)
plt.close(fig)